In [ ]:
import io
import zipfile
import requests
import frontmatter

In [11]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [12]:
moneyPrinterV2 = read_repo_data('FujiwaraChoki', 'MoneyPrinterV2')
print(f"moneyPrinterV2 documents: {len(moneyPrinterV2)}")


moneyPrinterV2 documents: 10


1. Simple Chunking


In [13]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result



In [14]:
moneyPrinter_chunks = []

for doc in moneyPrinterV2:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    moneyPrinter_chunks.extend(chunks)

print(f"Total chunks: {len(moneyPrinter_chunks)}")

Total chunks: 24


2. Splitting by Paragraphs and Sections


In [15]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        header = parts[i] + parts[i+1]
        header = header.strip()

        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections



In [16]:
moneyPrinter_sections = []

for doc in moneyPrinterV2:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        moneyPrinter_sections.append(section_doc)

print(f"Total sections: {len(moneyPrinter_sections)}")

Total sections: 38


3. LLM-based Intelligent Chunking

In [30]:
from dotenv import load_dotenv
load_dotenv()

True

In [41]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))


def llm(prompt, model='models/gemini-2.5-flash'):
    model_instance = genai.GenerativeModel(model)
    response = model_instance.generate_content(prompt)
    return response.text

In [42]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

In [43]:
# List available Gemini models
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(f"Model: {model.name}")
        print(f"  Display name: {model.display_name}")
        print()

Model: models/gemini-2.5-flash
  Display name: Gemini 2.5 Flash

Model: models/gemini-2.5-pro
  Display name: Gemini 2.5 Pro

Model: models/gemini-2.0-flash
  Display name: Gemini 2.0 Flash

Model: models/gemini-2.0-flash-001
  Display name: Gemini 2.0 Flash 001

Model: models/gemini-2.0-flash-lite-001
  Display name: Gemini 2.0 Flash-Lite 001

Model: models/gemini-2.0-flash-lite
  Display name: Gemini 2.0 Flash-Lite

Model: models/gemini-2.5-flash-preview-tts
  Display name: Gemini 2.5 Flash Preview TTS

Model: models/gemini-2.5-pro-preview-tts
  Display name: Gemini 2.5 Pro Preview TTS

Model: models/gemma-3-1b-it
  Display name: Gemma 3 1B

Model: models/gemma-3-4b-it
  Display name: Gemma 3 4B

Model: models/gemma-3-12b-it
  Display name: Gemma 3 12B

Model: models/gemma-3-27b-it
  Display name: Gemma 3 27B

Model: models/gemma-3n-e4b-it
  Display name: Gemma 3n E4B

Model: models/gemma-3n-e2b-it
  Display name: Gemma 3n E2B

Model: models/gemini-flash-latest
  Display name: Gemini

In [44]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [46]:
from tqdm.auto import tqdm
import time

moneyPrinter_llm_chunks = []

for i, doc in enumerate(tqdm(moneyPrinterV2)):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        moneyPrinter_llm_chunks.append(section_doc)
    
    # Rate limiting: Google free tier allows 5 requests/minute
    # Wait 13 seconds between documents (5 docs per minute)
    if i < len(moneyPrinterV2) - 1:
        time.sleep(13)

print(f"Total LLM chunks: {len(moneyPrinter_llm_chunks)}")

  0%|          | 0/10 [00:00<?, ?it/s]

Total LLM chunks: 68


In [47]:
moneyPrinter_llm_chunks 

[{'filename': 'MoneyPrinterV2-main/AGENTS.md',
  'section': '## Project Structure and Module Organization\n\n- `src/` contains the application code. Use `src/main.py` as the interactive entrypoint.\n- `src/classes/` holds provider-specific components (for example `YouTube.py`, `Twitter.py`, `Tts.py`, `AFM.py`, `Outreach.py`).\n- Shared utilities and configuration live in modules like `src/config.py`, `src/utils.py`, `src/cache.py`, and `src/constants.py`.\n- `scripts/` contains helper workflows such as setup, preflight checks, and upload helpers.\n- `docs/` contains feature documentation; `assets/` and `fonts/` contain static resources.'},
 {'filename': 'MoneyPrinterV2-main/AGENTS.md',
  'section': '## Development Workflow and Commands\n\n- `bash scripts/setup_local.sh`: bootstrap local development (creates `venv`, installs deps, seeds `config.json`, runs preflight).\n- `source venv/bin/activate && pip install -r requirements.txt`: manual dependency install/update.\n- `python3 scripts/